# Lending Club — Raw to Interim

**Purpose**
- Load the raw Lending Club file as published
- Inspect `loan_status` and vintage bad-rate patterns
- Materialize a clean interim DuckDB snapshot for all downstream notebooks

**About the dataset**
- Public release by Lending Club (peer-to-peer lending marketplace, active 2007-2020)
- Loan-level records for all *accepted* (originated) personal loans, 2007 through 2018 Q4
- Standard fields: loan amount, term, interest rate, grade/sub-grade, borrower financials (income, DTI, employment), loan purpose, state, and `loan_status` (repayment outcome)
- Lending Club also separately publishes a *rejected applications* file — not present in this project's `data/01_raw/` (see Scope below)

**Files in `data/01_raw/`**

| File | Size | Contents |
|---|---|---|
| `accepted_2007_to_2018Q4.csv.gz` | ~392 MB (gzip) | All accepted loans, 2007-2018 Q4, ~2.26M rows |

**Pipeline stages**

| Folder | Role |
|---|---|
| `data/01_raw/` | Untouched source file |
| `data/02_interim/` | This notebook's output — `raw_mat`, `matured`, `windowed` tables |
| `data/03_processed/` | Final modeling table, built later by `03_data_cleaning/01_cleaning_and_feature_prep.ipynb` |

**Scope**
- This notebook: accepted loans only
- Rejected-applications file: no `loan_status`/outcome → can't feed `is_bad` or `windowed`, so not loaded
- Reject inference (applied vs. approved population) is a separate, not-yet-built analysis — flagged as a scope decision, not an oversight

**Cell map**

| # | Step | Output |
|---|---|---|
| 1 | Connect + materialize `raw_mat` | Raw row count |
| 2 | `loan_status` breakdown | Status counts |
| 3 | Define `matured` + `is_bad` | Matured count, bad rate |
| 4 | Bad rate by origination year | Year-by-year table |
| 5 | Apply 2013-2017 window → `windowed` | Final row count |
| 6 | Summary | Table row counts, next notebook |

**Output:** `data/02_interim/lendingclub.duckdb` → `raw_mat`, `matured`, `windowed`

## Cell 1 — Connect & materialize raw file

- Querying the gzipped CSV directly via a DuckDB `VIEW` re-decompresses 392MB on every query
- Materialize once into `raw_mat` → downstream cells/notebooks pay that cost exactly once
- Load as `all_varchar=True`: raw file mixes numeric columns with stray text (e.g. "n/a") — type inference would silently drop/corrupt rows. Casting happens deliberately, per column, later in EDA notebooks

**Answers:** how many rows does the raw file actually contain?

In [9]:
import os
import duckdb

RAW_FILE = "../../data/01_raw/accepted_2007_to_2018Q4.csv.gz"
DUCKDB_FILE = "../../data/02_interim/lendingclub.duckdb"
ASSETS_TABLES = "../../data/04_assets/tables"
ASSETS_PLOTS = "../../data/04_assets/plots"

os.makedirs(os.path.dirname(DUCKDB_FILE), exist_ok=True)
os.makedirs(ASSETS_TABLES, exist_ok=True)
os.makedirs(ASSETS_PLOTS, exist_ok=True)

if os.path.exists(DUCKDB_FILE):
    os.remove(DUCKDB_FILE)

con = duckdb.connect(DUCKDB_FILE)

# read raw CSV as all-text (no type inference) -- avoids silently dropping/corrupting messy numeric columns
con.sql(f"""
    CREATE OR REPLACE VIEW raw AS
    SELECT * FROM read_csv('{RAW_FILE}', header=True, delim=',',
                            ignore_errors=True, all_varchar=True)
""")
# materialize once so later cells/notebooks don't re-decompress the gzip on every query
con.sql("CREATE OR REPLACE TABLE raw_mat AS SELECT * FROM raw")

n_raw = con.sql("SELECT count(*) FROM raw_mat").fetchone()[0]
print(f"raw file: {n_raw:,} rows")

**Result**
- Raw file: **2,260,701 rows** — full published Lending Club accepted-loan history
- Stored in `raw_mat`; no further gzip reads needed downstream

**Next:** inspect `loan_status` values.

## Cell 2 — What does `loan_status` actually contain?

- Look at actual raw values, not assumed categories
- Determines which loans are usable (finished outcome) vs. still open (unknown outcome)

**Answers:** which distinct status values exist, and how many rows fall into each?

In [10]:
# raw status counts, most frequent first
print("loan_status breakdown:")
status_breakdown = con.sql(
    "SELECT loan_status, count(*) AS n FROM raw_mat GROUP BY 1 ORDER BY 2 DESC"
).df()
print(status_breakdown.to_string(index=False))
status_breakdown.to_csv(os.path.join(ASSETS_TABLES, "ing01_status_breakdown.csv"), index=False)

loan_status breakdown:
                                        loan_status       n
                                         Fully Paid 1076751
                                            Current  878317
                                        Charged Off  268559
                                 Late (31-120 days)   21467
                                    In Grace Period    8436
                                  Late (16-30 days)    4349
 Does not meet the credit policy. Status:Fully Paid    1988
Does not meet the credit policy. Status:Charged Off     761
                                            Default      40
                                               None      33


**Result**

| loan_status | n |
|---|---|
| Fully Paid | 1,076,751 |
| Current | 878,317 |
| Charged Off | 268,559 |
| Late (31-120 days) | 21,467 |
| In Grace Period | 8,436 |
| Late (16-30 days) | 4,349 |
| Does not meet credit policy: Fully Paid | 1,988 |
| Does not meet credit policy: Charged Off | 761 |
| Default | 40 |
| NaN | 33 |

- **Finished (usable):** Fully Paid, Charged Off, Default, both "does not meet credit policy" variants
- **Still open (not usable):** Current, Late (both), In Grace Period — outcome not yet known (see `02_eda/07_target_outcome_objective.ipynb`)

In [11]:
# the 33 NaN-status rows -- inspect directly rather than assume what they are
print("the 33 rows where loan_status is NaN:")
sample = con.sql("SELECT * FROM raw_mat WHERE loan_status IS NULL LIMIT 5").df()
non_empty_cols = [c for c in sample.columns if sample[c].notna().any()]
print(f"columns with anything in them, out of {len(sample.columns)} total: {non_empty_cols}")
print(sample[non_empty_cols].to_string(index=False))
sample[non_empty_cols].to_csv(os.path.join(ASSETS_TABLES, "ing01_null_status_rows.csv"), index=False)

the 33 rows where loan_status is NaN:
columns with anything in them, out of 151 total: ['id']
                                              id
Total amount funded in policy code 1: 6417608175
Total amount funded in policy code 2: 1944088810
Total amount funded in policy code 1: 1741781700
 Total amount funded in policy code 2: 564202131
Total amount funded in policy code 1: 1791201400


**What the 33 `NaN` rows are**
- Every field empty except `id`, which holds text like `"Total amount funded in policy code 1: 6417608175"`
- CSV footer/summary lines, not loan records
- `read_csv(..., ignore_errors=True)` let them through as all-null rows instead of rejecting them
- They fall out of `matured` automatically (`NULL` status matches nothing in `MATURED_STATUSES`) — now documented explicitly rather than an implicit side effect

**Why open loans (`Current`/`Late`/`In Grace Period`) aren't modeled**

| Reason | Detail |
|---|---|
| Outcome unknown | A loan paying on time today could still default later — labeling it "good" now would be a guess, not a measurement |
| Future use: scoring | Once a model exists, score these loans for live risk (scoring needs features, not a known label) |
| Future use: monitoring | Track whether today's Late/Grace-Period loans cure or default — validates the labeling scheme itself |

Explored further in `02_eda/07_target_outcome_objective.ipynb`, cell 2.

**Next:** define matured loans and the `is_bad` target.

## Cell 3 — Define matured loans & `is_bad` target

- `is_bad = 1`: Charged Off, Default, "does not meet credit policy: Charged Off"
- `is_bad = 0`: Fully Paid, "does not meet credit policy: Fully Paid"
- This exact definition drives every downstream IV/WOE/model number — defined once, here

**Answers:** how many loans have reached a final outcome, and what's the bad rate among them?

In [12]:
MATURED_STATUSES = (
    "Fully Paid", "Charged Off", "Default",
    "Does not meet the credit policy. Status:Charged Off",
    "Does not meet the credit policy. Status:Fully Paid",
)
BAD_STATUSES = (
    "Charged Off", "Default",
    "Does not meet the credit policy. Status:Charged Off",
)
matured_list = ", ".join(f"'{s}'" for s in MATURED_STATUSES)
bad_list = ", ".join(f"'{s}'" for s in BAD_STATUSES)

# keep only finished-outcome loans, attach the is_bad label
con.sql(f"""
    CREATE OR REPLACE TABLE matured AS
    SELECT *, CASE WHEN loan_status IN ({bad_list}) THEN 1 ELSE 0 END AS is_bad
    FROM raw_mat
    WHERE loan_status IN ({matured_list})
""")
n_matured, n_bad = con.sql("SELECT count(*), sum(is_bad) FROM matured").fetchone()
print(f"matured (finished) loans: {n_matured:,} of {n_raw:,}")
print(f"bad rate among matured loans: {n_bad / n_matured:.2%}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

matured (finished) loans: 1,348,099 of 2,260,701
bad rate among matured loans: 19.98%


**Result**

| Metric | Value |
|---|---|
| Matured loans | 1,348,099 of 2,260,701 (59.6%) |
| Overall bad rate | 19.98% |

This bad rate is the baseline every downstream "bad rate by X" table is compared against.

**Next:** check whether every origination year is reliable to model on.

## Cell 4 — Bad rate by origination year

- Checking volume and bad rate per year rather than assuming the full history is usable
- Early years may be too low-volume to trust; the most recent vintage may be right-censored (not enough time for slow defaults to show up yet)

**Answers:** which years have enough volume and enough time-to-mature to be usable?

In [13]:
# bad rate & volume per origination year -- checks for low-volume years and right-censoring
print("bad rate by origination year:")
by_year = con.sql("""
    SELECT substr(issue_d, -4) AS year, count(*) AS n, round(avg(is_bad), 3) AS bad_rate
    FROM matured GROUP BY 1 ORDER BY 1
""").df()
print(by_year.to_string(index=False))
by_year.to_csv(os.path.join(ASSETS_TABLES, "ing01_by_year.csv"), index=False)

bad rate by origination year:
year      n  bad_rate
2007    603     0.262
2008   2393     0.207
2009   5281     0.137
2010  12537     0.140
2011  21721     0.152
2012  53367     0.162
2013 134804     0.156
2014 223103     0.184
2015 375546     0.202
2016 293105     0.233
2017 169321     0.231
2018  56318     0.158


**Result**

| Year | n | Bad rate |
|---|---|---|
| 2007 | 603 | 26.2% |
| 2008 | 2,393 | 20.7% |
| 2009 | 5,281 | 13.7% |
| 2010 | 12,537 | 14.0% |
| 2011 | 21,721 | 15.2% |
| 2012 | 53,367 | 16.2% |
| 2013 | 134,804 | 15.6% |
| 2014 | 223,103 | 18.4% |
| 2015 | 375,546 | 20.2% |
| 2016 | 293,105 | 23.3% |
| 2017 | 169,321 | 23.1% |
| 2018 | 56,318 | 15.8% |

- 2007-2012: too thin to trust
- 2018: sits well below the 2015-2017 trend → right-censoring, not a real quality improvement
- Explored further in `02_eda/06_temporal_sequential_spatial.ipynb`

**Next:** apply the modeling window.

## Cell 5 — Apply the modeling window (2013-2017)

- Recent enough to reflect current underwriting practice
- Old enough that every loan has had its full term to reach a final outcome

In [14]:
VINTAGE_START, VINTAGE_END = 2013, 2017

# restrict matured loans to the reliable modeling window
con.sql(f"""
    CREATE OR REPLACE TABLE windowed AS
    SELECT * FROM matured
    WHERE CAST(substr(issue_d, -4) AS INT) BETWEEN {VINTAGE_START} AND {VINTAGE_END}
""")
n_windowed = con.sql("SELECT count(*) FROM windowed").fetchone()[0]
print(f"rows in {VINTAGE_START}-{VINTAGE_END} window: {n_windowed:,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows in 2013-2017 window: 1,195,879


**Result**

| Metric | Value |
|---|---|
| Windowed rows | 1,195,879 of 1,348,099 matured (88.7%) |

`windowed` is the modeling population every EDA and cleaning notebook reads from here forward.

**Next:** summary.

## Cell 6 — Summary

- Print final table row counts and file path — facts only

In [15]:
# final table counts -- confirms what's available downstream
n_tables = {}
for tbl in ["raw_mat", "matured", "windowed"]:
    n_tables[tbl] = con.sql(f"SELECT count(*) FROM {tbl}").fetchone()[0]

print(f"file: {os.path.abspath('../../data/02_interim/lendingclub.duckdb')}")
for tbl, n in n_tables.items():
    print(f"{tbl}: {n:,} rows")
con.close()

file: e:\Repos\credit-risk-portfolio\phase0_data_platform\01_lendingclub\data\02_interim\lendingclub.duckdb
raw_mat: 2,260,701 rows
matured: 1,348,099 rows
windowed: 1,195,879 rows


**Output: `data/02_interim/lendingclub.duckdb`**

| Table | Rows |
|---|---|
| `raw_mat` | 2,260,701 |
| `matured` | 1,348,099 |
| `windowed` | 1,195,879 (19.98% bad rate) |

**Next notebook:** `02_eda/01_data_understanding_structural_profiling.ipynb`